
<font color=#009afa size=50>Blockchain.com API</font>

<font size=50>DAT5002 - Blockchain Fundamentals</font>

|**Assessment Code:** |PRAC1|
|---|---|
|**Author:**|Tom Rowan, ST20285213|
|**Course:**| DAT5002_S2_25|
|**Course Tutor:**| Dr. Ali Shahaab|

In [1]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
from matplotlib.pyplot import figure
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Datetime
from datetime import datetime

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Import My Blockchain Libraries
import blockchainDataAPI.bcda as bcda
import blockchainDataAPI.bcblock as bcblock


  __  _____   ____   ___ ____  ___    ____ ____  _ _____           _____                    __                          
 / _\/__   \ |___ \ / _ \___ \( _ )  | ___|___ \/ |___ /          /__   \___  _ __ ___     /__\ _____      ____ _ _ __  
 \ \   / /\/   __) | | | |__) / _ \  |___ \ __) | | |_ \   _____    / /\/ _ \| '_ ` _ \   / \/// _ \ \ /\ / / _` | '_ \ 
 _\ \ / /     / __/| |_| / __/ (_) |  ___) / __/| |___) | |_____|  / / | (_) | | | | | | / _  \ (_) \ V  V / (_| | | | |
 \__/ \/     |_____|\___/_____\___/  |____/_____|_|____/           \/   \___/|_| |_| |_| \/ \_/\___/ \_/\_/ \__,_|_| |_|

 BlockChain.com Data API Module

 


## Task 2.1

i) At the time of your writing, what is the height of the most recent block mined and what is the 
difficulty level? (5%)

In [4]:
# Create BC Data API object
#bc=bcda.BlockchainComDataAPI(verbose=True)
bc=bcda.BlockchainComDataAPI()

# Fetch data on the latest block to be mined into a Block object
latest_block_json = bc.latest_block()
if "status" not in latest_block_json.keys():
    the_block = bcblock.Block(latest_block_json['hash'])
else:
    print (latest_block_json)


Block object initialised


The height of the block is returned by the API natively, and the Block object simply returns this:

In [5]:
print (Fore.CYAN + "Height: " + Style.RESET_ALL + str(the_block.height()) )


Height: 921161


The number of leading zeroes required for the target hash of a block is an indicator of the difficulty of generating that hash. As the number of zeroes increases, the difficulty also increases.

In [6]:
print (Fore.CYAN + "Hash: " + Style.RESET_ALL + the_block.hash() )
print (Fore.CYAN + "Number of leading zeroes: " + Style.RESET_ALL + str(the_block.leading_zeroes()))

Hash: 00000000000000000000d3496dc2f06b0770d98e939420d883834f3739a9fa76
Number of leading zeroes: 20


There is also a calculation for the difficulty of the block:

In [7]:
print (Fore.CYAN + "Difficulty: " + Style.RESET_ALL + the_block.difficulty())

Difficulty: [Warning - Not Implemented]


ii) What is the address of the miner of the block ? Can you unmask the identity of the 
address? (5%)

In [8]:
the_block.mined_by()

'SpiderPool [1BM1sAcrfV6d4zPKytzziu4McLQDsFC2Qc]'

iii) How many transactions does it contain? (5%)

In [9]:
print (Fore.CYAN + "Number of Transactions: " + Style.RESET_ALL + str(the_block.number_tx()))

Number of Transactions: 3795


iv) Which address has the most balance (5%)

In [10]:
tx_list = the_block.tx()

len(tx_list)

3795

In [11]:


addresses = []

for tx in tx_list:
    inputs = tx['inputs']
    outputs = tx['out']

    for o in outputs:
        if 'addr' in o.keys():
            _addr = o['addr']
            addresses.append(_addr)

print (len(addresses))

# Deduplicate bye cast to a set and back
addresses = list(set(addresses))

print (len(addresses))

6685
1193


We need to use blocks of 50 addresses to balance the URL max length and the number of requests.

In [ ]:
def chunk_list(the_list, size=50):
    return [the_list[i:i + size] for i in range(0, len(the_list), size)]

join_with_pipe = lambda the_list: "|".join(map(str, the_list))


# Chunk the data
chunks = chunk_list(addresses)

max_balance = { 
               'address': 'blah',
               'balance': -1
               }

for chunk in chunks:

    address_string = join_with_pipe(chunk)
    balances = bc.balance(address_string)

    for the_address in balances:
        the_balance = balances[the_address]['final_balance']
        if the_balance > 0:
            if the_balance > max_balance['balance']:
                max_balance['balance'] = the_balance
                max_balance['address'] = the_address
            
            # We might find a duplicate value, and should call that out
            elif the_balance == max_balance['balance']:
                print ("Duplicate value found: ", max_balance['address'], max_balance['balance'], the_address, the_balance)

In [ ]:
# Balance is in Satoshi, so convert to whole Bitcoin
max_balance['btc_val'] = (max_balance['balance']/10e8)
print (Fore.CYAN + "Max Balance in block: " + Style.RESET_ALL +  str(max_balance['btc_val']) + " btc [" + max_balance['address'] + "]")

Max Balance in block: 1903.622459838 btc [bc1qr4dl5wa7kl8yu792dceg9z5knl2gkn220lk7a9]


v) Which address has most number of transactions. (5%)

In [15]:
max_tx = { 
            'address': 'blah',
            'number_tx': -1
        }

# We can reuse the chunks
for chunk in chunks:

    address_string = join_with_pipe(chunk)
    records = bc.multi_address(address_string)['addresses']

    for record in records:

        num_tx = record['n_tx']
        if num_tx > 0:
            if num_tx  > max_tx['number_tx']:
                max_tx['number_tx'] = num_tx
                max_tx['address'] = record['address']
            
            # We might find a duplicate value, and should call that out
            elif num_tx == max_tx['number_tx']:
                print ("Duplicate value found: ", max_tx['address'], max_tx['number_tx'], record['address'], num_tx)


In [16]:
print (Fore.CYAN + "Max number Txs in block: " + Style.RESET_ALL +  str(max_tx['number_tx']) + " [" + max_tx['address'] + "]")

Max number Txs in block: 2643824 [bc1q7cyrfmck2ffu2ud3rn5l5a8yv6f0chkp0zpemf]


vi) When did this address become active (first transaction on the network) (5)?

In [ ]:
#offset = "offset=" + str(max_tx['number_tx'])
record = bc.single_address(max_tx['address'], offset=max_tx['number_tx'])

def timestamp_to_time(timestamp):
    _datetime = datetime.fromtimestamp(timestamp)
    _mined_at = _datetime.strftime("%d/%m/%Y %H:%M")
    return _mined_at

first_tx_index = len(record['txs'])

print (first_tx_index)

first_tx = record['txs'][first_tx_index-1]

timestamp_to_time(first_tx['time'])


https://blockchain.info/rawaddr/bc1q7cyrfmck2ffu2ud3rn5l5a8yv6f0chkp0zpemf?format=json&2643824
100


'28/10/2025 08:51'

In [21]:
record

{'hash160': 'f60834ef165253c571b11ce9fa74e46692fc5ec1',
 'address': 'bc1q7cyrfmck2ffu2ud3rn5l5a8yv6f0chkp0zpemf',
 'n_tx': 2643825,
 'n_unredeemed': 299,
 'total_received': 2643945227158132,
 'total_sent': 2643827202786181,
 'final_balance': 118024371951,
 'txs': [{'hash': '45c71c4f693373cffd544c32525a09d630588aeb88cedd9ccf037dc406dd4c7a',
   'ver': 2,
   'vin_sz': 1,
   'vout_sz': 2,
   'size': 223,
   'weight': 562,
   'fee': 432,
   'relayed_by': '0.0.0.0',
   'lock_time': 0,
   'tx_index': 4303057417965472,
   'double_spend': False,
   'time': 1761646302,
   'block_index': None,
   'block_height': None,
   'inputs': [{'sequence': 4294967293,
     'witness': '02483045022100c0a19db466da882804e859fbd2e18264f9fc1967044a29f609c71788966b3c7002201efbbbe6063056ecb8df26e8c48adb765177b4e5c75b91db6589df55c0c619560121026e5628506ecd33242e5ceb5fdafe4d3066b5c0f159b3c05a621ef65f177ea286',
     'script': '',
     'index': 0,
     'prev_out': {'type': 0,
      'spent': True,
      'value': 102972809

## Task 2.2 

Given the information of the above block, how can you retrieve all bitcoin blockchain data up to 1st Jan 2020? Briefly explain your answer. (No code required, just approach or pseudocode is 
sufficient). (Max 300 words) (5%)